# 05 - Full Cross-Project Evaluation (39 assigned pairs)

Runs the CNN regressor from 04 over the 39 training/testing pairs assigned in the project list
(PS-3, BERT word embeddings, cross-project, random split). Per pair: 5-fold CV on the source
train pool, final model trained on the full pool, single evaluation on the target test set
(same seed-42 partitions as the median baseline, so numbers are directly comparable).

The 39 pairs use the project names from the assignment PDF. They are mapped to the TAWOS project
names; one documented mapping: the PDF's "C++ driver (Appcelerator)" corresponds to TAWOS
"Aptana Studio" (Aptana Studio is the Appcelerator project in the dataset).

Outputs: results_pairs.csv, evaluation_summary.txt, results plots.

In [ ]:
import glob, os, re, time
import numpy as np, pandas as pd

csv = glob.glob("/kaggle/input/**/all_issues_sp.csv", recursive=True)[0]
df = pd.read_csv(csv)
df["storypoint"] = pd.to_numeric(df["storypoint"], errors="coerce")
df = df.dropna(subset=["storypoint"])
df = df[(df.storypoint > 0) & (df.storypoint <= 40)]
df["text"] = df["text"].fillna("").astype(str)
df["project"] = df["project"].astype(str).str.strip()   # some TAWOS names have trailing spaces

# identical split construction to 02_baseline_median (seed 42, same rng consumption order),
# converted to per-project local row ranks to index the embedding files
rng = np.random.default_rng(42)
splits = {}
for proj, g in df.groupby("project"):
    idx = g.index.to_numpy()
    rng.shuffle(idx)
    n_test = max(1, int(round(0.2 * len(idx))))
    local = np.arange(len(g))
    lmap = dict(zip(g.index.to_numpy().tolist(), local.tolist()))
    splits[proj] = dict(test=np.array([lmap[v] for v in idx[:n_test]]),
                        train=np.array([lmap[v] for v in idx[n_test:]]))

EMB = glob.glob("/kaggle/input/**/emb", recursive=True)[0]
META = EMB.replace("/emb", "/meta")
def safe(n):
    return re.sub(r"[^A-Za-z0-9_]", "_", str(n))

_files = sorted(os.path.basename(p)[:-4] for p in glob.glob(f"{EMB}/*.npy"))
def resolve(name):
    c = safe(name)
    if c in _files:
        return c
    m = [f for f in _files if f.rstrip("_") == c.rstrip("_")]
    assert m, f"no embedding file for {name!r} (looked for {c})"
    return m[0]

_cache = {}
def load_project(name):
    if name not in _cache:
        key = resolve(name)
        e = np.load(f"{EMB}/{key}.npy", mmap_mode="r")
        m = pd.read_csv(f"{META}/{key}.csv")
        _cache[name] = (e, m)
    return _cache[name]

base = pd.read_csv(glob.glob("/kaggle/input/**/baseline_results.csv", recursive=True)[0])
base["source"] = base["source"].astype(str).str.strip()
base["target"] = base["target"].astype(str).str.strip()
within = base[base.target.str.contains("WITHIN")].set_index("source")
for proj in list(splits)[:5]:
    assert len(splits[proj]["test"]) == int(within.loc[proj, "n_test"]), proj
print("splits OK, match baseline sizes")

In [ ]:
import torch, torch.nn as nn

MAX_LEN, D = 128, 768

class CNNRegressor(nn.Module):
    def __init__(self, d_in=D, ch=(256, 128, 64), p=0.2):
        super().__init__()
        c1, c2, c3 = ch
        self.conv = nn.Sequential(
            nn.Conv1d(d_in, c1, 3, padding=1), nn.ReLU(), nn.Dropout(p),
            nn.Conv1d(c1, c2, 3, padding=1), nn.ReLU(),
            nn.Conv1d(c2, c3, 3, padding=1), nn.ReLU(),
        )
        self.head = nn.Sequential(nn.Linear(c3, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x, ntok):
        h = self.conv(x.transpose(1, 2))
        mask = (torch.arange(x.size(1), device=x.device)[None, :] < ntok[:, None]).unsqueeze(1)
        h = h.masked_fill(~mask, float("-inf"))
        return self.head(h.max(dim=2).values).squeeze(-1)

def batch_to_gpu(X, y, ntok):
    return (torch.tensor(X, dtype=torch.float32, device="cuda"),
            torch.tensor(y, dtype=torch.float32, device="cuda"),
            torch.tensor(ntok, dtype=torch.long, device="cuda"))

def train_model(Xtr, ytr, ntr, epochs=25, bs=64, lr=1e-3, patience=4, seed=0, Xva=None, yva=None, nva=None):
    torch.manual_seed(seed); np.random.seed(seed)
    model = CNNRegressor().cuda()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.L1Loss()
    n = len(ytr); best = (1e9, None, 0); hist = []
    for ep in range(epochs):
        model.train()
        perm = np.random.permutation(n)
        for b0 in range(0, n, bs):
            sel = perm[b0:b0 + bs]
            x, y, nt = batch_to_gpu(Xtr[sel], ytr[sel], ntr[sel])
            with torch.autocast("cuda", dtype=torch.float16):
                loss = lossf(model(x, nt), y)
            opt.zero_grad(); loss.backward(); opt.step()
        if Xva is not None:
            model.eval(); errs = []
            with torch.no_grad():
                for b0 in range(0, len(yva), 256):
                    x, y, nt = batch_to_gpu(Xva[b0:b0+256], yva[b0:b0+256], nva[b0:b0+256])
                    errs.append((model(x, nt) - y).abs().cpu().numpy())
            vmae = float(np.concatenate(errs).mean()); hist.append(vmae)
            if vmae < best[0]: best = (vmae, {k: t.clone() for k, t in model.state_dict().items()}, ep)
            if ep - best[2] >= patience: break
    if Xva is not None: model.load_state_dict(best[1])
    return model, hist

def predict(model, X, ntok, bs=512):
    model.eval(); out = []
    with torch.no_grad():
        for b0 in range(0, len(X), bs):
            x = torch.tensor(X[b0:b0+bs], dtype=torch.float32, device="cuda")
            nt = torch.tensor(ntok[b0:b0+bs], dtype=torch.long, device="cuda")
            out.append(model(x, nt).cpu().numpy())
    return np.concatenate(out)

def run_pair(src, tgt, seed=42):
    Xs, ms = load_project(src); Xt, mt = load_project(tgt)
    tr_pos, te_pos = splits[src]["train"], splits[tgt]["test"]
    Xtr, ytr = Xs[tr_pos], ms.storypoint.values[tr_pos]
    ntr = np.minimum(ms.n_tokens.values[tr_pos], MAX_LEN)
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    fold_maes, best_eps = [], []
    for f, (a, b) in enumerate(kf.split(np.arange(len(ytr)))):
        mdl, hist = train_model(Xtr[a], ytr[a], ntr[a], seed=seed + f,
                                Xva=Xtr[b], yva=ytr[b], nva=ntr[b])
        fold_maes.append(hist[-1]); best_eps.append(len(hist))
    fix_epochs = int(np.clip(round(np.mean(best_eps)), 5, 25))
    mdl, _ = train_model(Xtr, ytr, ntr, epochs=fix_epochs, seed=seed)
    pred = predict(mdl, Xt[te_pos], np.minimum(mt.n_tokens.values[te_pos], MAX_LEN))
    ytest = mt.storypoint.values[te_pos]
    err = np.abs(pred - ytest)
    return dict(val_mae_mean=float(np.mean(fold_maes)), val_mae_std=float(np.std(fold_maes)),
                epochs=fix_epochs, test_mae=float(err.mean()), test_mdae=float(np.median(err)))

from sklearn.model_selection import KFold
print("model + run_pair ready")

In [ ]:
# the 39 assigned pairs, PDF names, in the order of the assignment table
PAIRS = [
    ("XD", "MESOS"), ("Usergrid", "MXNet"), ("Titanium Mobile Platform", "Usergrid"),
    ("The Titanium SDK", "Alloy Framework"), ("Software Cloud", "Appcelerator Daemon"),
    ("Sawtooth", "Appcelerator Studio"), ("Nexus", "Command-Line Interface"),
    ("MXNet", "The Titanium SDK"), ("Mule APIkit", "Titanium Mobile Platform"),
    ("Mule", "C++ driver (Appcelerator)"), ("Moodle", "Atlassian Software Server"),
    ("MongoDB Core Server", "Bamboo"), ("MESOS", "Clover"),
    ("Lyrasis Dura Cloud", "Confluence Cloud"), ("Lsstcorp Data management", "Confluence Server"),
    ("Jira Server", "Crowd"), ("Jira Cloud", "FishEye"), ("Java driver", "Jira Cloud"),
    ("Indy SDK", "Jira Server"), ("Indy Node", "Software Cloud"),
    ("FishEye", "DotNetNuke Platform"), ("Fabric", "Blockchain Explorer"),
    ("Evergreen", "Fabric"), ("DotNetNuke Platform", "Indy Node"), ("DataCass", "Indy SDK"),
    ("Crowd", "Sawtooth"), ("Confluence Server", "Lsstcorp Data management"),
    ("Confluence Cloud", "Lyrasis Dura Cloud"), ("Compass", "MongoDB Core Server"),
    ("Command-Line Interface", "Compass"), ("Clover", "C++ driver (MongoDB)"),
    ("C++ driver (Appcelerator)", "Evergreen"), ("C++ driver (MongoDB)", "Java driver"),
    ("Blockchain Explorer", "Moodle"), ("Bamboo", "Mule"),
    ("Atlassian Software Server", "Mule APIkit"), ("Appcelerator Studio", "Nexus"),
    ("Appcelerator Daemon", "DataCass"), ("Alloy Framework", "XD"),
]
NAME = {
    "XD": "Spring XD", "MESOS": "Apache Mesos", "Usergrid": "Apache Usergrid", "MXNet": "Apache MXNet",
    "The Titanium SDK": "The Titanium SDK", "Software Cloud": "Atlassian Software Cloud",
    "Sawtooth": "Hyperledger Sawtooth", "Nexus": "Sonatype Nexus", "Bamboo": "Atlassian Bamboo",
    "Clover": "Atlassian Clover", "Confluence Cloud": "Atlassian Confluence Cloud",
    "Confluence Server": "Atlassian Confluence Server", "Jira Server": "Atlassian Jira Server",
    "Crowd": "Atlassian Crowd", "Jira Cloud": "Atlassian Jira Cloud", "FishEye": "Atlassian FishEye",
    "Java driver": "The Mongo Java driver", "Indy SDK": "Hyperledger Indy SDK",
    "Indy Node": "Hyperledger Indy Node", "Fabric": "Hyperledger Fabric",
    "Blockchain Explorer": "Hyperledger Blockchain Explorer", "DataCass": "Spring DataCass",
    "Compass": "MongoDB Compass", "C++ driver (MongoDB)": "The Mongo C++ driver",
    "Evergreen": "The MongoDB Engineering", "C++ driver (Appcelerator)": "Aptana Studio",
    "Atlassian Software Server": "Atlassian Atlassian Software Server",
}
def dsn(n):
    return NAME.get(n, n)

base_idx = base.set_index(["source", "target"])
rows = []
t0 = time.time()
for i, (s, t) in enumerate(PAIRS):
    src, tgt = dsn(s), dsn(t)
    try:
        r = run_pair(src, tgt)
    except Exception as ex:
        print(f"[{i+1}/39] {s} -> {t}: SKIPPED - {type(ex).__name__}: {ex}", flush=True)
        rows.append(dict(pdf_source=s, pdf_target=t, source=src, target=tgt, status="missing"))
        pd.DataFrame(rows).to_csv("/kaggle/working/results_pairs.csv", index=False)
        continue
    brow = base_idx.loc[(src, tgt)]
    r.update(pdf_source=s, pdf_target=t, source=src, target=tgt, status="ok",
             baseline_mae=float(brow.mae), baseline_mdae=float(brow.mdae))
    rows.append(r)
    pd.DataFrame(rows).to_csv("/kaggle/working/results_pairs.csv", index=False)
    print(f"[{i+1}/39] {s} -> {t}: test MAE {r['test_mae']:.3f} (baseline {r['baseline_mae']:.3f}), "
          f"MdAE {r['test_mdae']:.3f}, val {r['val_mae_mean']:.3f}+/-{r['val_mae_std']:.3f} "
          f"| {time.time()-t0:.0f}s", flush=True)
print("ALL PAIRS DONE in", round((time.time()-t0)/60, 1), "min")

In [ ]:
res = pd.DataFrame(rows)
ok = res[res.status == "ok"]
nmiss = int((res.status == "missing").sum())
tmae, bmae = ok.test_mae, ok.baseline_mae
lines = [
    "PS-3 cross-project evaluation - BERT word embeddings + 3-layer CNN",
    "seed 42 | SP in (0,40] | same 80:20 partitions as baseline | 5-fold CV on source train pool",
    "",
    f"pairs evaluated: {len(ok)} of {len(res)} assigned (missing data: {nmiss})",
    f"test MAE  across pairs: {tmae.mean():.3f} +/- {tmae.std():.3f}  (baseline {bmae.mean():.3f} +/- {bmae.std():.3f})",
    f"test MdAE across pairs: {res.test_mdae.mean():.3f}              (baseline {res.baseline_mdae.mean():.3f})",
    f"beats baseline MAE on {int((tmae < bmae).sum())}/{len(ok)} pairs",
    f"5-fold val MAE mean over pairs: {res.val_mae_mean.mean():.3f} +/- {res.val_mae_std.mean():.3f}",
    "",
    "best 5 pairs by test MAE:",
    ok.nsmallest(5, "test_mae")[["pdf_source", "pdf_target", "test_mae", "test_mdae", "baseline_mae"]].to_string(index=False),
    "",
    "worst 5 pairs by test MAE:",
    ok.nlargest(5, "test_mae")[["pdf_source", "pdf_target", "test_mae", "test_mdae", "baseline_mae"]].to_string(index=False),
]
txt = chr(10).join(lines)
open("/kaggle/working/evaluation_summary.txt", "w").write(txt)
print(txt)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(19, 5))
x = np.arange(len(res))
ax.bar(x - 0.2, bmae, 0.4, label="median baseline")
ax.bar(x + 0.2, tmae, 0.4, label="BERT + CNN")
ax.set_xticks(x)
ax.set_xticklabels([f"{s[:9]}>{t[:9]}" for s, t in zip(ok.pdf_source, ok.pdf_target)],
                   rotation=90, fontsize=6)
ax.axhline(bmae.mean(), color="gray", ls=":")
ax.set_ylabel("test MAE"); ax.legend()
ax.set_title("Cross-project test MAE per assigned pair")
plt.tight_layout(); plt.savefig("/kaggle/working/results_per_pair.png", dpi=130); plt.show()